In [1]:

import numpy as np
import tensorflow as tf
import os

# Thu muc goc cua project: D:\vietnamese-banknote-cnn
PROJECT_DIR = os.path.abspath("../..")

MODEL_PATH = os.path.join(PROJECT_DIR, "models", "cnn_model.h5")
WEIGHTS_DIR = os.path.join(PROJECT_DIR, "weights")

os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Load dung model da kiem tra
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

# Lay trong so tu model
conv1_w, conv1_b = model.layers[0].get_weights()
conv2_w, conv2_b = model.layers[1].get_weights()
dense_w, dense_b = model.layers[4].get_weights()

# Chuyen sang Q8.8
SCALE = 256

conv1_w_q = np.round(conv1_w * SCALE).astype(np.int16)
conv1_b_q = np.round(conv1_b * SCALE).astype(np.int16)

conv2_w_q = np.round(conv2_w * SCALE).astype(np.int16)
conv2_b_q = np.round(conv2_b * SCALE).astype(np.int16)

dense_w_q = np.round(dense_w * SCALE).astype(np.int16)
dense_b_q = np.round(dense_b * SCALE).astype(np.int16)

print("Model loaded:", MODEL_PATH)
print("Conv1:", conv1_w.shape, conv1_b.shape)
print("Conv2:", conv2_w.shape, conv2_b.shape)
print("Dense:", dense_w.shape, dense_b.shape)
print("Q8.8 conversion done.")


Model loaded: d:\vietnamese-banknote-cnn\models\cnn_model.h5
Conv1: (3, 3, 1, 8) (8,)
Conv2: (3, 3, 8, 8) (8,)
Dense: (1568, 1) (1,)
Q8.8 conversion done.


In [2]:

def save_mem_hex(filename, array):
    flat = array.flatten()

    with open(filename, "w", encoding="utf-8") as f:
        for val in flat:
            v = int(val) & 0xFFFF
            f.write(f"{v:04x}\n")

    print(f"Saved: {filename} ({len(flat)} values)")


In [3]:

save_mem_hex(os.path.join(WEIGHTS_DIR, "conv1_w.mem"), conv1_w_q)
save_mem_hex(os.path.join(WEIGHTS_DIR, "conv1_b.mem"), conv1_b_q)

save_mem_hex(os.path.join(WEIGHTS_DIR, "conv2_w.mem"), conv2_w_q)
save_mem_hex(os.path.join(WEIGHTS_DIR, "conv2_b.mem"), conv2_b_q)

save_mem_hex(os.path.join(WEIGHTS_DIR, "dense_w.mem"), dense_w_q)
save_mem_hex(os.path.join(WEIGHTS_DIR, "dense_b.mem"), dense_b_q)

print("\nExport completed.")
print("Conv1 max abs:", np.max(np.abs(conv1_w_q)))
print("Conv2 max abs:", np.max(np.abs(conv2_w_q)))
print("Dense max abs:", np.max(np.abs(dense_w_q)))
print("Bias max abs:", max(
    np.max(np.abs(conv1_b_q)),
    np.max(np.abs(conv2_b_q)),
    np.max(np.abs(dense_b_q))
))


Saved: d:\vietnamese-banknote-cnn\weights\conv1_w.mem (72 values)
Saved: d:\vietnamese-banknote-cnn\weights\conv1_b.mem (8 values)
Saved: d:\vietnamese-banknote-cnn\weights\conv2_w.mem (576 values)
Saved: d:\vietnamese-banknote-cnn\weights\conv2_b.mem (8 values)
Saved: d:\vietnamese-banknote-cnn\weights\dense_w.mem (1568 values)
Saved: d:\vietnamese-banknote-cnn\weights\dense_b.mem (1 values)

Export completed.
Conv1 max abs: 89
Conv2 max abs: 93
Dense max abs: 52
Bias max abs: 25


In [4]:

# Kiem tra dau ra Conv1 bang so nguyen Q8.8
import numpy as np

data = np.load(os.path.join(PROJECT_DIR, "data", "data.npy"))
labels = np.load(os.path.join(PROJECT_DIR, "data", "labels.npy"))

# Chon anh dau tien, nhan 0 = tien 20k
img = data[0]
img_q = np.round(img * SCALE).astype(np.int16)

H, W = 32, 32
OUT = 30
FILTERS = 8

conv1_out_q = np.zeros((OUT, OUT, FILTERS), dtype=np.int16)

for f in range(FILTERS):
    for i in range(OUT):
        for j in range(OUT):
            acc = 0
            for ki in range(3):
                for kj in range(3):
                    pixel = int(img_q[i + ki, j + kj, 0])
                    weight = int(conv1_w_q[ki, kj, 0, f])
                    acc += pixel * weight

            # Chuyen tich luy ve Q8.8
            acc = acc >> 8
            acc += int(conv1_b_q[f])

            conv1_out_q[i, j, f] = np.int16(acc)

print("Image label:", labels[0])
print("Image shape:", img.shape)
print("Conv1 output shape:", conv1_out_q.shape)
print("First 10 outputs, filter 0:")
print(conv1_out_q[:, :, 0].flatten()[:10])
print("Min output:", conv1_out_q.min())
print("Max output:", conv1_out_q.max())


Image label: 0
Image shape: (32, 32, 1)
Conv1 output shape: (30, 30, 8)
First 10 outputs, filter 0:
[25 26 26 27 26 26 27 28 28 26]
Min output: -53
Max output: 372


In [5]:
# Luu dau ra Conv1 cua Python de doi chieu voi Verilog
import os
import numpy as np

RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

conv1_ref_path = os.path.join(RESULTS_DIR, "conv1_python_ref.mem")

with open(conv1_ref_path, "w", encoding="utf-8") as f:
    for value in conv1_out_q[:, :, 0].flatten():
        f.write(f"{int(value) & 0xFFFF:04x}\n")

print("Da luu:", conv1_ref_path)
print("So gia tri:", conv1_out_q[:, :, 0].size)
print("10 gia tri dau:", conv1_out_q[:, :, 0].flatten()[:10].tolist())

Da luu: d:\vietnamese-banknote-cnn\results\conv1_python_ref.mem
So gia tri: 900
10 gia tri dau: [25, 26, 26, 27, 26, 26, 27, 28, 28, 26]


In [6]:

import os
import numpy as np

data = np.load(os.path.join(PROJECT_DIR, "data", "data.npy"))

img = data[0, :, :, 0]
img_q = np.round(img * SCALE).astype(np.int16)

pixel_path = os.path.join(RESULTS_DIR, "conv1_pixels.mem")

with open(pixel_path, "w", encoding="utf-8") as f:
    for value in img_q[0:3, 0:3].flatten():
        f.write(f"{int(value) & 0xFFFF:04x}\n")

print("Saved:", pixel_path)
print("9 pixels Q8.8:")
print(img_q[0:3, 0:3].flatten().tolist())


Saved: d:\vietnamese-banknote-cnn\results\conv1_pixels.mem
9 pixels Q8.8:
[79, 72, 69, 78, 70, 68, 75, 71, 70]


In [9]:

# Kiem tra tung buoc tinh Conv1 tai vi tri (0, 0), filter 0

pixels = img_q[0:3, 0:3].flatten().astype(np.int64)
weights = conv1_w_q[:, :, 0, 0].flatten().astype(np.int64)
products = pixels * weights

print("Pixels:", pixels.tolist())
print("Weights:", weights.tolist())
print("Products:", products.tolist())
print("Sum products:", int(products.sum()))
print("Sum after shift:", int(products.sum()) >> 8)
print("Bias:", int(conv1_b_q[0]))
print(
    "Conv1 before ReLU:",
    (int(products.sum()) >> 8) + int(conv1_b_q[0])
)


Pixels: [79, 72, 69, 78, 70, 68, 75, 71, 70]
Weights: [7, 80, 37, -16, 32, -23, -39, -35, -39]
Products: [553, 5760, 2553, -1248, 2240, -1564, -2925, -2485, -2730]
Sum products: 154
Sum after shift: 0
Bias: 25
Conv1 before ReLU: 25


In [10]:

# Xuat toan bo anh 32x32 sang file .mem cho ModelSim

import os
import numpy as np

data = np.load(os.path.join(PROJECT_DIR, "data", "data.npy"))
img_q_full = np.round(data[0, :, :, 0] * SCALE).astype(np.int16)

image_path = os.path.join(RESULTS_DIR, "image32_q.mem")

with open(image_path, "w", encoding="utf-8") as f:
    for value in img_q_full.flatten():
        f.write(f"{int(value) & 0xFFFF:04x}\n")

print("Saved:", image_path)
print("So pixel:", img_q_full.size)
print("9 pixel dau:", img_q_full.flatten()[:9].tolist())


Saved: d:\vietnamese-banknote-cnn\results\image32_q.mem
So pixel: 1024
9 pixel dau: [79, 72, 69, 66, 65, 62, 60, 58, 59]


In [11]:

# Xuat dau ra Conv1 cua ca 8 bo loc de doi chieu Verilog

import os
import numpy as np

# Tinh lai dau ra Conv1 cho ca 8 filter
img_q = np.round(data[0] * SCALE).astype(np.int16)

conv1_ref_all = np.zeros((30, 30, 8), dtype=np.int16)

for f in range(8):
    for i in range(30):
        for j in range(30):
            acc = 0

            for ki in range(3):
                for kj in range(3):
                    pixel = int(img_q[i + ki, j + kj, 0])
                    weight = int(conv1_w_q[ki, kj, 0, f])
                    acc += pixel * weight

            acc = acc >> 8
            acc += int(conv1_b_q[f])
            conv1_ref_all[i, j, f] = np.int16(acc)

ref_all_path = os.path.join(RESULTS_DIR, "conv1_python_all.mem")

with open(ref_all_path, "w", encoding="utf-8") as f:
    for value in conv1_ref_all.flatten():
        f.write(f"{int(value) & 0xFFFF:04x}\n")

print("Saved:", ref_all_path)
print("Output shape:", conv1_ref_all.shape)
print("Total values:", conv1_ref_all.size)
print("First 8 values:", conv1_ref_all[0, 0, :].tolist())
print("Filter 0 first value:", int(conv1_ref_all[0, 0, 0]))


Saved: d:\vietnamese-banknote-cnn\results\conv1_python_all.mem
Output shape: (30, 30, 8)
Total values: 7200
First 8 values: [25, 26, 121, 30, 19, 44, 20, 5]
Filter 0 first value: 25


In [13]:
import os
import numpy as np

# Conv1 trước đó đã được tính và lưu trong conv1_ref_all
# Kích thước: (30, 30, 8)

conv1_relu_all = np.maximum(conv1_ref_all, 0).astype(np.int16)

relu_path = os.path.join(RESULTS_DIR, "conv1_relu_all.mem")
with open(relu_path, "w", encoding="utf-8") as file:
    for value in conv1_relu_all.flatten():
        file.write(f"{int(value) & 0xFFFF:04x}\n")

print("Saved:", relu_path)
print("Output shape:", conv1_relu_all.shape)
print("Total values:", conv1_relu_all.size)
print("First 8 values:", conv1_relu_all[0, 0, :].tolist())
print("Negative values remaining:", int(np.sum(conv1_relu_all < 0)))

Saved: d:\vietnamese-banknote-cnn\results\conv1_relu_all.mem
Output shape: (30, 30, 8)
Total values: 7200
First 8 values: [25, 26, 121, 30, 19, 44, 20, 5]
Negative values remaining: 0


In [14]:
# Tính Conv2 bằng số nguyên Q8.8 để đối chiếu với Verilog
conv2_ref_all = np.zeros((28, 28, 8), dtype=np.int16)

for r in range(28):
    for c in range(28):
        for f in range(8):
            acc = 0

            for ki in range(3):
                for kj in range(3):
                    for ch in range(8):
                        pixel = int(conv1_relu_all[r + ki, c + kj, ch])
                        weight = int(conv2_w_q[ki, kj, ch, f])
                        acc += pixel * weight

            # Chuyển tích Q16.16 về Q8.8, sau đó cộng bias
            acc = acc >> 8
            acc += int(conv2_b_q[f])

            conv2_ref_all[r, c, f] = np.int16(acc)

conv2_path = os.path.join(RESULTS_DIR, "conv2_python_all.mem")
with open(conv2_path, "w", encoding="utf-8") as file:
    for value in conv2_ref_all.flatten():
        file.write(f"{int(value) & 0xFFFF:04x}\n")

print("Saved:", conv2_path)
print("Output shape:", conv2_ref_all.shape)
print("Total values:", conv2_ref_all.size)
print("First 8 values:", conv2_ref_all[0, 0, :].tolist())

Saved: d:\vietnamese-banknote-cnn\results\conv2_python_all.mem
Output shape: (28, 28, 8)
Total values: 6272
First 8 values: [51, 44, 58, 47, 14, 15, 16, 32]


In [15]:
import numpy as np
from pathlib import Path

ROOT = Path(r"D:\vietnamese-banknote-cnn")
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def read_mem_signed16(path):
    values = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            value = int(line, 16)

            if value >= 32768:
                value -= 65536

            values.append(value)

    return np.array(values, dtype=np.int32)

def write_mem_signed16(path, values):
    with open(path, "w") as f:
        for value in np.asarray(values).flatten():
            f.write(f"{int(value) & 0xFFFF:04x}\n")

# Doc ket qua Conv2 da kiem tra PASS
conv2 = read_mem_signed16(
    ROOT / "conv2_python_all.mem"
)

assert conv2.size == 6272, "Conv2 phai co 6272 gia tri"

# Conv2: 28x28x8
conv2 = conv2.reshape(28, 28, 8)

# ReLU
conv2_relu = np.maximum(conv2, 0)

# MaxPooling 2x2
maxpool = np.zeros((14, 14, 8), dtype=np.int32)

for r in range(14):
    for c in range(14):
        for ch in range(8):
            patch = conv2_relu[
                r*2:r*2+2,
                c*2:c*2+2,
                ch
            ]

            maxpool[r, c, ch] = np.max(patch)

# Xuat file
write_mem_signed16(
    RESULTS_DIR / "maxpool_python_all.mem",
    maxpool
)

print("MaxPool shape:", maxpool.shape)
print("Total values:", maxpool.size)
print("PASS: Export MaxPool reference completed.")

MaxPool shape: (14, 14, 8)
Total values: 1568
PASS: Export MaxPool reference completed.


In [16]:
import shutil

shutil.copy(
    ROOT / "results" / "maxpool_python_all.mem",
    ROOT / "maxpool_python_all.mem"
)

print("Copied maxpool_python_all.mem")

Copied maxpool_python_all.mem


In [17]:
import numpy as np
from pathlib import Path

ROOT = Path(r"D:\vietnamese-banknote-cnn")

# Doc 1568 gia tri MaxPooling
values = []

with open(ROOT / "maxpool_python_all.mem", "r") as f:
    for line in f:
        value = int(line.strip(), 16)

        if value >= 32768:
            value -= 65536

        values.append(value)

# MaxPooling output: 14x14x8
maxpool = np.array(values, dtype=np.int32).reshape(14, 14, 8)

# Flatten
flatten = maxpool.flatten()

assert flatten.size == 1568

print("MaxPool shape:", maxpool.shape)
print("Flatten shape:", flatten.shape)
print("Total values:", flatten.size)
print("PASS: Flatten completed.")

MaxPool shape: (14, 14, 8)
Flatten shape: (1568,)
Total values: 1568
PASS: Flatten completed.


In [18]:
import numpy as np
from pathlib import Path
import shutil

ROOT = Path(r"D:\vietnamese-banknote-cnn")
WEIGHTS = ROOT / "weights"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

def read_mem(path):
    values = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            value = int(line, 16)
            if value >= 32768:
                value -= 65536
            values.append(value)
    return np.array(values, dtype=np.int64)

def write_mem(path, values):
    with open(path, "w") as f:
        for value in np.asarray(values).flatten():
            f.write(f"{int(value) & 0xFFFF:04x}\n")

# Doc du lieu
x = read_mem(ROOT / "maxpool_python_all.mem")
w = read_mem(WEIGHTS / "dense_w.mem")
b = read_mem(WEIGHTS / "dense_b.mem")

assert len(x) == 1568
assert len(w) == 1568
assert len(b) == 1

# Dense Q8.8
acc = int(np.dot(x, w))
dense_full = (acc >> 8) + int(b[0])

# Giu 16 bit thap giong Verilog
dense_q = ((dense_full + 32768) % 65536) - 32768

write_mem(RESULTS / "dense_python_ref.mem", [dense_q])

shutil.copy(
    RESULTS / "dense_python_ref.mem",
    ROOT / "dense_python_ref.mem"
)

print("Accumulator:", acc)
print("Dense full:", dense_full)
print("Dense Q8.8:", dense_q)
print("PASS: Dense reference exported.")

Accumulator: -36118
Dense full: -145
Dense Q8.8: -145
PASS: Dense reference exported.


In [19]:
import numpy as np

# Gia tri Dense Q8.8
dense_value = dense_q / 256.0

# Sigmoid
sigmoid = 1 / (1 + np.exp(-dense_value))

# Phan loai
threshold = 0.5

prediction = 1 if sigmoid >= threshold else 0

print("Dense:", dense_value)
print("Sigmoid:", sigmoid)
print("Threshold:", threshold)

if prediction == 0:
    print("Prediction: 20.000 VND")
else:
    print("Prediction: 50.000 VND")

Dense: -0.56640625
Sigmoid: 0.3620664767467021
Threshold: 0.5
Prediction: 20.000 VND
